In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_ANSWER = r"(epsilon - f*d)/(2*k_B*ln(g))"

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def normalize_expr(text):
    if text is None:
        return ""

    s = str(text)

    # Remove label
    s = re.sub(r"^\s*T_c\s*=\s*", "", s)

    # Unicode normalization
    unicode_map = {
        "ε": "epsilon",
        "ϵ": "epsilon",
        "−": "-",
        "–": "-",
        "×": "*",
        "·": "*",
    }
    for k, v in unicode_map.items():
        s = s.replace(k, v)

    # LaTeX normalization
    s = s.replace("\\epsilon", "epsilon")
    s = s.replace("\\ln", "ln")
    s = s.replace("\\log", "ln")
    s = s.replace("log", "ln")
    s = s.replace("k_B", "kB")
    s = s.replace("k_{B}", "kB")

    # Remove spaces
    s = re.sub(r"\s+", "", s)

    # Normalize multiplication
    s = s.replace("f*d", "fd").replace("d*f", "fd").replace("df", "fd")

    # Normalize ln(g)
    s = s.replace("ln(g)", "lng")

    # Normalize k → kB
    s = re.sub(r"\bk\b", "kB", s)

    return s.lower()

# ----------------------------
# Strict code verifier
# ----------------------------
def code_verifier(answer_text):
    s = normalize_expr(answer_text)

    if not s:
        return False, "hallucination"

    # Correct structure
    correct = (
        "epsilon-fd" in s and
        "2*kb*lng" in s or "2kb*lng" in s
    )

    # Explicit wrong pattern
    if "2fd" in s:
        return False, "misapplication_of_equation_or_model"

    if correct:
        return True, None

    return None, None  # uncertain

# ----------------------------
# LLM semantic verifier (fallback)
# ----------------------------
def llm_semantic_verifier(judge_llm, submitted):
    judge_prompt = f"""
You are a strict physics answer verifier.

Determine if the submitted expression is mathematically equivalent to the canonical answer.

Return JSON only:
{{
  "equivalent": true or false,
  "failure_mode": "<one of: failure_to_recognize_key_aspects, hallucination, misapplication_of_equation_or_model, incorrect_factual_knowledge, calculation_error>",
  "reason": "<short reason>"
}}

Canonical:
{CANONICAL_ANSWER}

Submitted:
{submitted}
"""

    resp = judge_llm.prompt(judge_prompt)
    parsed = extract_json(resp)

    if parsed is None:
        return False, "hallucination"

    return parsed.get("equivalent", False), parsed.get("failure_mode", "hallucination")

# ----------------------------
# Trace builder
# ----------------------------
def build_trace(**kwargs):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        **kwargs
    }

# ----------------------------
# Task
# ----------------------------
@kbench.task(
    name="FP-0002 DNA Zipper Critical Temperature",
    description="Symbolic statistical-physics reasoning with entropy and mechanical work contributions."
)
def fp_0002_dna_zipper_critical_temperature(llm) -> tuple[int, int]:

    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A toy model for two complementary strands of DNA resembles a zipper. The two strands are connected by links spaced at equal intervals $d$. It costs energy $\epsilon$ to break a link. Each broken link produces two dangling ends, each with $g$ internal states.

A force $f$ is applied to pull the strands apart.

Question: What is the critical temperature $T_c(g,\epsilon,f,d)$?

Return JSON:
{
  "final_answer": "<expression>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")

        # 1. Code verifier
        code_result, code_failure = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1

        elif code_result is False:
            failure_mode = code_failure

        else:
            # 2. LLM fallback
            equivalent, llm_failure = llm_semantic_verifier(llm, final_answer)

            if equivalent:
                passed_checks = 1
            else:
                failure_mode = llm_failure

    trace = build_trace(
        task_id="fp_0002",
        model=str(llm),
        pass_result=(passed_checks == 1),
        final_answer=final_answer,
        normalized=normalize_expr(final_answer),
        failure_mode=failure_mode,
        raw_output=response
    )

    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0002_dna_zipper_critical_temperature.run(kbench.llm)

In [ ]:
results = fp_0002_dna_zipper_critical_temperature.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0002"]